# Google Colab
This notebook is intended to be executed on Google Golab.

In [1]:
# setup for Google Colab
!pip install torch torchvision
!pip install requests
!pip install tqdm

In [2]:
# imports
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import requests
import subprocess

In [31]:
class MyDataset():
    def __init__(self, dataset='places2', split='train'):
        self.__root_dir = Path().cwd() # /content on google colab
        self.__datasets = {
            'places2': {
                'name': 'places2',
                'url': 'https://ctipub-my.sharepoint.com/:u:/g/personal/marius_vlad_dumitru_stud_etti_upb_ro/IQDIXTjG7MdBRrMizEF0KjaWAR46T_nR4YNLuvv3HThlCN4?download=1',
                'archive': self.__root_dir / 'places2_reduced.tar',
                'extract_dir': self.__root_dir / 'places2'
            },
            'tinyimagenet': {
                'name': 'tinyimagenet',
                'url': 'https://ctipub-my.sharepoint.com/:u:/g/personal/marius_vlad_dumitru_stud_etti_upb_ro/IQBkCh4mNvQ8Q7t70sitsGOyAWD29CnOXxthhkGPnbmeRBY?download=1',
                'archive': self.__root_dir / 'tinyimagenet_flat.tar',
                'extract_dir': self.__root_dir / 'tinyimagenet'
            }
        }

        self.change_split(split)
        self.change_dataset(dataset)

    def __download_dataset(self):
        try:
            print(f"[INFO] Downloading dataset {self.__dataset['name']}. This may take a whike ...")
            response = requests.get(self.__dataset['url'], stream=True)
            response.raise_for_status()
            total_size = int(response.headers.get("Content-Length", 0))
            with open(str(self.__dataset['archive']), "wb") as f, tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                desc="Downloading",
            ) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
            print(f"[INFO] Download complete: {str(self.__dataset['archive'])}")
            return True

        except Exception as e:
            print(f"[ERROR] Download failed: {e}")
            return False
    
    def __extract_dataset(self):
        try:
            print(f"[INFO] Extracting dataset {self.dataset['archive']}. This may take a while ...")
            archive = self.__dataset['archive']
            out_dir = self.__dataset['extract_dir']
            out_dir.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["tar", "-xf", str(archive), "-C", str(out_dir)],
                check=True
            )

            print(f"[INFO] Extraction complete: {out_dir}")
            return True

        except Exception as e:
            print(f"[ERROR] Extraction failed: {e}")
            return False
    
    def change_dataset(self, new_dataset):
        self.__dataset = self.__datasets[new_dataset] 
        if not self.__dataset['archive'].exists():
            self.__download_dataset()
        if not self.__dataset['extract_dir'].exists():
            self.__extract_dataset()

        self.__train_dir = self.__dataset['extract_dir'] / 'train'
        self.__val_dir = self.__dataset['extract_dir'] / 'val'

    def get_dataset(self):
        return self.__dataset['name']
        
    def change_split(self, new_split):
        self.__split = new_split

    def get_split(self):
        return self.__split

    def __len__(self):
        pass
    
    def __getitem__(self):
        pass

In [ ]:
# Creating the mydataset object
mydataset = MyDataset(dataset='tinyimagenet')
print(f'Current dataset is {mydataset.get_dataset()}')
print(f'Current split is {mydataset.get_split()}')

Current dataset is tinyimagenet
Current split is train


In [33]:
# Changing the dataset
mydataset.change_dataset('places2')
print(f'Current dataset is {mydataset.get_dataset()}')
print(f'Current split is {mydataset.get_split()}')

Current dataset is places2
Current split is train


In [34]:
# Changing split
mydataset.change_split('val')
print(f'Current split is {mydataset.get_split()}')

Current split is val


In [ ]:
!rm -r ./*
!ls -la

rm: cannot remove './*': No such file or directory
total 16
drwxr-xr-x 1 root root 4096 Mar 25 22:43 .
drwxr-xr-x 1 root root 4096 Mar 25 21:57 ..
drwxr-xr-x 4 root root 4096 Mar 23 13:29 .config
